*Rent_model*

In [27]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

# --- (辅助函数 ) ---
def parse_first_number(s, default_val=np.nan):
    # 提取字符串中的第一个数字, 处理 '暂无'
    if pd.isnull(s): return default_val
    s = str(s).replace('暂无', '0')
    # 处理范围，例如 '1.1-1.85' -> 1.475
    range_match = re.match(r'(\d+\.?\d*)-(\d+\.?\d*)', s)
    if range_match:
        return (float(range_match.group(1)) + float(range_match.group(2))) / 2.0
    # 提取单个数字
    match = re.search(r'(\d+\.?\d*)', s)
    return float(match.group(1)) if match else default_val

def parse_rent_term(s):
    # 解析 85 种租期格式为平均月数
    if pd.isnull(s): return np.nan
    s = str(s)
    # 替换年为月
    s = s.replace('1年以内', '0~12个月').replace('2年以内', '0~24个月')
    s = s.replace('3年以内', '0~36个月').replace('3年以上', '36~60个月') # 估算上限
    s = s.replace('1年以上', '12~24个月').replace('6个月以上', '6~12个月') # 估算
    s = s.replace('8个月以上', '8~12个月').replace('4个月以上', '4~12个月')
    s = s.replace('2个月以上', '2~12个月').replace('1个月以上', '1~12个月')
    s = s.replace('5个月以上', '5~12个月').replace('11个月以上', '11~12个月')
    s = s.replace('10个月以上', '10~12个月').replace('9个月以上', '9~12个月')
    s = s.replace('1年', '12个月').replace('2年', '24个月').replace('3年', '36个月')
    s = s.replace('1~2年', '12~24个月').replace('1~3年', '12~36个月')
    
    numbers = re.findall(r'(\d+)', s)
    if not numbers: return np.nan
    nums = [int(n) for n in numbers]
    
    # 根据关键字估算或计算平均值
    if '以上' in s: # (已替换大部分, 保留以防万一)
        return nums[0] + 6 
    if '以内' in s: # (已替换大部分)
        return nums[0] / 2
        
    return np.mean(nums) # 处理 "X个月" 或 "X~Y个月"

In [28]:
# --- STAGE 1: 加载并进行通用（非统计性）预处理 ---
print("--- STAGE 1: 加载并进行通用预处理... ---")
df = pd.read_csv('comment_combined_rent.csv',encoding='gbk') 

#2.删除不需要的列
df.drop(['环线位置','年份','开发商','物业公司',
         '物业办公电话','coord_x','coord_y','装修'
        ],axis=1,inplace=True)

--- STAGE 1: 加载并进行通用预处理... ---


In [ ]:
# --- STAGE 2: 加载并进行通用（非统计性）预处理 ---
# 面积
df['面积'] = df['面积'].astype(str).str.replace('㎡', '', regex=False)
df['面积'] = pd.to_numeric(df['面积'], errors='coerce')
# 交易时间
if '交易时间' in df.columns: df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')

# 付款方式-清除http
dirty_mask = df['付款方式'].astype(str).str.contains('http', na=False)
df.loc[dirty_mask, '付款方式'] = np.nan

#电梯&燃气映射
df['电梯'] = df['电梯'].replace({'有': 1, '无': 0})
df['燃气'] = df['燃气'].replace({'有': 1, '无': 0})

# 租期
df['租期_月数'] = df['租期'].apply(parse_rent_term)

# 建筑年代转化成均值
def parse_build_year(s):
    if pd.isnull(s): return np.nan
    s = str(s); years = re.findall(r'(\d{4})', s)
    if not years: return np.nan
    return np.mean([float(y) for y in years]) if len(years) > 1 else float(years[0])
df['建筑年代'] = df['建筑年代'].apply(parse_build_year)

#房屋总数&楼栋总数
df['房屋总数'] = pd.to_numeric(df['房屋总数'].astype(str).str.replace('户', '', regex=False), errors='coerce')
df['楼栋总数'] = pd.to_numeric(df['楼栋总数'].astype(str).str.replace('栋', '', regex=False), errors='coerce')

# 绿化率百分比字符串（如 '35%'）转换为数值比例（如 0.35）。
df['绿化率'] = pd.to_numeric(df['绿 化 率'].astype(str).str.replace('%', '', regex=False), errors='coerce') / 100.0

#容积率
df['容积率'] = pd.to_numeric(df['容 积 率'], errors='coerce')

#物业费&燃气费&供热费等均值处理
def parse_range_mean(s):
    if pd.isnull(s):
        return np.nan

    s = str(s).strip() # 转换为字符串并移除空格，使用正则表达式查找所有浮点数或整数：\d+ 匹配一个或多个数字 \.? 匹配零个或一个点 \d* 匹配零个或多个数字（在点后面）
    numbers = re.findall(r'(\d+\.?\d*)', s)

    if not numbers:
        return np.nan
    float_numbers = [float(n) for n in numbers]

    # 如果找到多个数字，返回它们的平均值；否则返回找到的唯一值
    if len(float_numbers) > 1:
        return np.mean(float_numbers)
    else:
        return float_numbers[0]
    
df['物业费'] = df['物 业 费'].apply(parse_range_mean)
df['燃气费'] = df['燃气费'].apply(parse_range_mean)
df['供热费'] = df['供热费'].apply(parse_range_mean)

#停车费
def get_first_number(s):
    if pd.isnull(s):
        return np.nan

    s = str(s)
    match = re.search(r'(\d+\.?\d*)', s)

    if match:
        return float(match.group(0))
    else:
        return np.nan
df['停车费用'] = df['停车费用'].apply(get_first_number)

#计算距离
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    计算两点之间的 Haversine 距离 (单位：公里)
    """
    R = 6371  # 地球半径 (公里)
    
    # 确保输入是浮点数并处理潜在的NaN
    try:
        lon1, lat1, lon2, lat2 = map(float, [lon1, lat1, lon2, lat2])
    except (ValueError, TypeError):
        return np.nan # 如果任何一个是NaN或无法转换，返回NaN
        
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    distance = R * c
    return distance

In [ ]:
# --- STAGE 3: 填充区县、板块缺失数据 ---

from sklearn.cluster import KMeans

# --- 辅助函数：安全获取众数 ---
def get_safe_mode(series):
    """
    获取 Series 的众数，如果为空则返回 NaN。
    """
    mode = series.mode()
    return mode.iloc[0] if not mode.empty else np.nan

# --- 1. L1 - 按层级填充 (板块 < 区县 < 城市) ---
# (这是最高效的第一步)

# 1a. 填充 '板块' (使用其所在 '区县' 的众数)
print("--- L1: 正在按 '区县' 填充 '板块'... ---")
# (我们只在非空数据上计算规则)
sector_by_region_map = df[df['板块'].notnull()].groupby('区县')['板块'].apply(get_safe_mode).to_dict()
df['板块'] = df['板块'].fillna(df['区县'].map(sector_by_region_map))

# 1b. 填充 '区县' (使用其所在 '城市' 的众数)
print("--- L1: 正在按 '城市' 填充 '区县'... ---")
region_by_city_map = df[df['区县'].notnull()].groupby('城市')['区县'].apply(get_safe_mode).to_dict()
df['区县'] = df['区县'].fillna(df['城市'].map(region_by_city_map))

# --- 2. L2 - 按 K-Means 地理聚类填充 (Fallback) ---
# 检查 L1 之后是否仍有缺失
missing_mask = df['板块'].isnull() | df['区县'].isnull()

# (根据您的保证，我们假设 lon/lat 在 missing_mask 中都存在)
if missing_mask.any():
    print(f"--- L2: 仍有 {missing_mask.sum()} 行缺失位置。正在启动 K-Means 地理填充... ---")
    
    # 2a. 准备 K-Means 数据
    # 'known_data' = 我们有完整地理信息（lon, lat, 板块, 区县）的行
    known_data_mask = (
        df['lon'].notnull() & df['lat'].notnull() &
        df['板块'].notnull() & df['区县'].notnull()
    )
    known_data = df[known_data_mask].copy()
    
    # 'target_data' = 我们有 lon/lat，但 L1 填充失败的行
    # (根据您的保证，missing_mask 的所有行都有 lon/lat)
    target_data = df[missing_mask].copy() 

    if not known_data.empty and not target_data.empty:
        # 2b. 'Fit': 在 'known_data' 上拟合 K-Means
        k_clusters = 100 
        kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
        kmeans.fit(known_data[['lon', 'lat']])
        
        # 2c. 'Build Rules': 找出每个聚类的 '板块' 和 '区县' 众数
        known_data['cluster'] = kmeans.predict(known_data[['lon', 'lat']])
        cluster_sector_map = known_data.groupby('cluster')['板块'].apply(get_safe_mode).to_dict()
        cluster_region_map = known_data.groupby('cluster')['区县'].apply(get_safe_mode).to_dict()
        
        # 2d. 'Predict & Transform': 为 'target_data' 预测聚类并应用规则
        target_data['cluster'] = kmeans.predict(target_data[['lon', 'lat']])
        
        # 使用 K-Means 规则填充
        target_sector_fill = target_data['cluster'].map(cluster_sector_map)
        target_region_fill = target_data['cluster'].map(cluster_region_map)
        
        # 2e. 将填充值应用回原始 df
        df.loc[target_data.index, '板块'] = df.loc[target_data.index, '板块'].fillna(target_sector_fill)
        df.loc[target_data.index, '区县'] = df.loc[target_data.index, '区县'].fillna(target_region_fill)
        
        print(f"--- L2: K-Means 已应用于 {target_data.index.size} 行数据 ---")
        
        # (L3的全局兜底已被移除)
        # 检查是否还有 K-Means 无法填充的 (例如，某个聚类没有板块/区县众数)
        if df.loc[target_data.index, '板块'].isnull().any():
             print("--- L2 警告: K-Means 填充后 '板块' 仍有缺失，将使用全局众数兜底... ---")
             df['板块'] = df['板块'].fillna(df['板块'].mode()[0])
        if df.loc[target_data.index, '区县'].isnull().any():
             print("--- L2 警告: K-Means 填充后 '区县' 仍有缺失，将使用全局众数兜底... ---")
             df['区县'] = df['区县'].fillna(df['区县'].mode()[0])
             
    else:
        print("--- L2: K-Means 填充已跳过 (没有 'known_data' 或 'target_data') ---")
        # (如果跳过了，我们必须使用全局众数兜底)
        print("--- L2 跳过，使用 L3 全局众数兜底... ---")
        df['板块'] = df['板块'].fillna(df['板块'].mode()[0])
        df['区县'] = df['区县'].fillna(df['区县'].mode()[0])

print("\n--- '板块' 和 '区县' 填充完毕 ---")

--- L1: 正在按 '区县' 填充 '板块'... ---
--- L1: 正在按 '城市' 填充 '区县'... ---
--- L2: 仍有 6041 行缺失位置。正在启动 K-Means 地理填充... ---
--- L2: K-Means 已应用于 6041 行数据 ---

--- '板块' 和 '区县' 填充完毕 ---


In [ ]:
# --- STAGE 4: 填充其他缺失数据 ---

# (辅助函数：安全地获取众数)
def _get_mode(series):
    mode = series.mode()
    return mode.iloc[0] if not mode.empty else np.nan

# (辅助函数：应用规则)
def _apply_fill_rules(df, rules):
    df = df.copy()
    
    # 1. 填充数值型列
    num_cols = rules.get('numerical_cols', [])
    for col in num_cols:
        if '板块' in df.columns:
            df[col] = df[col].fillna(df['板块'].map(rules['sector_medians'].get(col, {})))
        if '区县' in df.columns:
            df[col] = df[col].fillna(df['区县'].map(rules['region_medians'].get(col, {})))
        if '城市' in df.columns:
            df[col] = df[col].fillna(df['城市'].map(rules['city_medians'].get(col, {})))
        df[col] = df[col].fillna(rules.get('global_medians', {}).get(col))
    
    # 2. 填充对象型列
    cat_cols = rules.get('categorical_cols', [])
    for col in cat_cols:
        if '板块' in df.columns:
            df[col] = df[col].fillna(df['板块'].map(rules['sector_modes'].get(col, {})))
        if '区县' in df.columns:
            df[col] = df[col].fillna(df['区县'].map(rules['region_modes'].get(col, {})))
        if '城市' in df.columns:
            df[col] = df[col].fillna(df['城市'].map(rules['city_modes'].get(col, {})))
        df[col] = df[col].fillna(rules.get('global_modes', {}).get(col))
                      
    return df

# --- 1. 拆分数据 ---
print("--- 1. 正在拆分数据... ---")
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 2. 填充分组键 (板块/区县/城市) ---
print("--- 2. 正在填充分组键 (板块/区县/城市)... ---")
group_keys = ['板块', '区县', '城市']
rules_group_keys = {}
for col in group_keys:
    if col in train_df.columns:
        mode_val = train_df[col].mode()[0]
        rules_group_keys[col] = mode_val # 存储规则
        train_df[col] = train_df[col].fillna(mode_val)
        test_df[col] = test_df[col].fillna(mode_val) 

# --- 3. 计算所有填充规则 (仅在 train_df 上) ---
print("--- 3. 正在 train_df 上计算所有填充规则... ---")
train_rules = {}

# 数值型规则
numerical_cols = ['电梯','燃气','建筑年代','房屋总数','楼栋总数','燃气费','供热费','停车位','停车费用','租期_月数','绿化率','容积率','物业费']
numerical_cols = [col for col in numerical_cols if col in train_df.columns]
train_rules['numerical_cols'] = numerical_cols
train_rules['global_medians'] = train_df[numerical_cols].median()
train_rules['sector_medians'] = train_df.groupby('板块')[numerical_cols].median()
train_rules['region_medians'] = train_df.groupby('区县')[numerical_cols].median()
train_rules['city_medians'] = train_df.groupby('城市')[numerical_cols].median()

# 对象型规则
categorical_cols = ['户型','楼层','朝向','付款方式','车位','用水','用电','采暖','配套设施','物业类别','建筑结构','产权描述','供水','供暖','供电']
categorical_cols = [col for col in categorical_cols if col in train_df.columns]
train_rules['categorical_cols'] = categorical_cols
train_rules['global_modes'] = train_df[categorical_cols].apply(_get_mode)
train_rules['sector_modes'] = train_df.groupby('板块')[categorical_cols].apply(_get_mode)
train_rules['region_modes'] = train_df.groupby('区县')[categorical_cols].apply(_get_mode)
train_rules['city_modes'] = train_df.groupby('城市')[categorical_cols].apply(_get_mode)

print("    ...规则计算完毕。")

# --- 4. 应用填充规则 ---
print("--- 4. 正在应用填充规则到 train_df 和 test_df... ---")
train_df_filled = _apply_fill_rules(train_df, train_rules)
test_df_filled = _apply_fill_rules(test_df, train_rules)

# --- 5. 计算并应用 Distance 特征 ---
print("--- 5. 正在计算和应用 'distance_to_center' 特征... ---")

# 'Fit': 只在训练集上计算中心
sector_centers = train_df_filled.groupby('板块')[['lon', 'lat']].mean().reset_index()
sector_centers.rename(columns={'lon': 'sector_center_lon', 'lat': 'sector_center_lat'}, inplace=True)
global_center_lon = train_df_filled['lon'].mean()
global_center_lat = train_df_filled['lat'].mean()

# 'Transform': 应用到 train_df_filled
train_df_filled = pd.merge(train_df_filled, sector_centers, on='板块', how='left')
train_df_filled['distance_to_center'] = train_df_filled.apply(
    lambda row: haversine_distance(row['lon'], row['lat'], row['sector_center_lon'], row['sector_center_lat']), 
    axis=1
)

# 'Transform': 应用到 test_df_filled
test_df_filled = pd.merge(test_df_filled, sector_centers, on='板块', how='left')
test_df_filled['sector_center_lon'].fillna(global_center_lon, inplace=True)
test_df_filled['sector_center_lat'].fillna(global_center_lat, inplace=True)
test_df_filled['distance_to_center'] = test_df_filled.apply(
    lambda row: haversine_distance(row['lon'], row['lat'], row['sector_center_lon'], row['sector_center_lat']), 
    axis=1
)

# 清理辅助列
train_df_filled.drop(['sector_center_lon', 'sector_center_lat'], axis=1, inplace=True)
test_df_filled.drop(['sector_center_lon', 'sector_center_lat'], axis=1, inplace=True)
print("    ...'distance_to_center' 特征已创建。")

# --- 6. 重新合并 ---
print("--- 6. 正在重新合并数据以便后续处理... ---")
df = pd.concat([train_df_filled, test_df_filled], ignore_index=True)

df.fillna({'室': 0, '厅': 0, '卫': 0}, inplace=True)

print(f"\n--- 流程完成。新的 'df' 维度: {df.shape} ---")

--- 1. 正在拆分数据... ---
--- 2. 正在填充分组键 (板块/区县/城市)... ---
--- 3. 正在 train_df 上计算所有填充规则... ---
    ...规则计算完毕。
--- 4. 正在应用填充规则到 train_df 和 test_df... ---
--- 5. 正在计算和应用 'distance_to_center' 特征... ---
    ...'distance_to_center' 特征已创建。
--- 6. 正在重新合并数据以便后续处理... ---

--- 流程完成。新的 'df' 维度: (108672, 45) ---


In [ ]:
# --- STAGE 5: 数据拆分 ---
# 户型 (解析)
if '户型' in df.columns:
    df['室'] = df['户型'].str.extract(r'(\d+)室').astype(float)
    df['厅'] = df['户型'].str.extract(r'(\d+)厅').astype(float)
    df['卫'] = df['户型'].str.extract(r'(\d+)卫').astype(float)
    df.loc[df['户型'].str.contains('房间', na=False), '室'] = df['户型'].str.extract(r'(\d+)房间', expand=False).astype(float)

# 朝向
if '朝向' in df.columns:
    df['朝南'] = df['朝向'].str.contains('南', na=False).astype(int)
    df['朝北'] = df['朝向'].str.contains('北', na=False).astype(int)
    df['朝东'] = df['朝向'].str.contains('东', na=False).astype(int)
    df['朝西'] = df['朝向'].str.contains('西', na=False).astype(int)

def process_and_engineer_floor_features(df, floor_col='楼层', group_col='板块'):
    """
    一个完整的函数，用于处理楼层数据并创建用于回归的特征。

    步骤:
    1. 将 '楼层' 拆分为 '总楼层' 和 '相对位置'。
    2. 使用 'group_col' (例如 '板块' 或 '小区') 的众数来填充 '总楼层' 的缺失值。
    3. 将 '相对位置' (数字和文本) 转换为统一的 '楼层_数值'。
    4. 计算 '楼层比例' (楼层_数值 / 总楼层)。
    """
    # --- 步骤 1: 拆分 ---
    
    floor_str = df[floor_col].astype(str)
    
    df['总楼层'] = np.nan
    df['相对位置'] = np.nan

    # 1a. 处理带斜杠的 (例如 "22/32层" 或 "中楼层/38层")
    has_slash = floor_str.str.contains('/', na=False)
    
    if has_slash.any():
        parts = floor_str[has_slash].str.split('/', n=1, expand=True)
        df.loc[has_slash, '相对位置'] = parts[0]
        df.loc[has_slash, '总楼层'] = pd.to_numeric(
            parts[1].str.extract(r'(\d+)', expand=False), 
            errors='coerce'
        )

    # 1b. 处理不带斜杠的 (例如 "地下4层" 或 "暂无数据")
    no_slash = ~has_slash
    if no_slash.any():
        df.loc[no_slash, '相对位置'] = floor_str[no_slash]
        # '总楼层' 保持为 NaN，等待下一步填充


    # --- 步骤 2: 用板块众数填充 '总楼层' 的缺失值 ---

    # .transform 会将分组计算的众数应用回原始的索引
    fill_value = df.groupby(group_col)['总楼层'].transform(
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
    df['总楼层'] = df['总楼层'].fillna(fill_value)


    # --- 步骤 3: 特征工程 (计算 '楼层_数值') ---

    # 临时列，用于统一计算
    df['楼层_数值'] = pd.to_numeric(df['相对位置'], errors='coerce')
    
    # 定义映射关系
    absolute_map = {
        '底层': 1,
        '地下室': -1
    }
    
    # 3a. 填充绝对值 (底层, 地下室)
    df['楼层_数值'] = df['楼层_数值'].fillna(df['相对位置'].map(absolute_map))

    # 3b. 填充相对值 (顶, 高, 中, 低)
    #     注意: 这里使用的是 步骤2 填充过的 '总楼层'
    total_fl = df['总楼层'] 
    is_nan = df['楼层_数值'].isna() # 找到还未被赋值的行

    df.loc[is_nan & (df['相对位置'] == '顶层'), '楼层_数值'] = total_fl
    df.loc[is_nan & (df['相对位置'] == '高楼层'), '楼层_数值'] = total_fl * 0.8
    df.loc[is_nan & (df['相对位置'] == '中楼层'), '楼层_数值'] = total_fl * 0.5
    df.loc[is_nan & (df['相对位置'] == '低楼层'), '楼层_数值'] = total_fl * 0.2

    # 3c. 填充 "地下X层" (例如 '地下4层')
    #     这些也是 is_nan 状态
    underground = (is_nan & df['相对位置'].astype(str).str.startswith('地下'))
    if underground.any():
        extracted_num = df.loc[underground, '相对位置'].str.extract(r'(\d+)')[0]
        df.loc[underground, '楼层_数值'] = pd.to_numeric(extracted_num, errors='coerce') * -1

    
    # --- 步骤 4: 计算 '楼层比例' ---
    # 使用 .replace(0, np.nan) 防止除以0
    df['楼层比例'] = df['楼层_数值'] / df['总楼层'].replace(0, np.nan)
    
    # (可选) 清理临时列
    df = df.drop(columns=['楼层_数值'])
    
    # (可选) 清理一下 '相对位置' 列中纯数字的情况
    numeric_pos = pd.to_numeric(df['相对位置'], errors='coerce')
    df['相对位置'] = numeric_pos.where(numeric_pos.notna(), df['相对位置'])

# 调用函数
process_and_engineer_floor_features(df, floor_col='楼层',  group_col='板块')

df['楼层比例'].fillna(df.groupby('板块')['楼层比例'].transform('median'), inplace=True)

#独热编码
# 注意 df 后面的 [[...]]
dummy_features = pd.get_dummies(
    df[['付款方式', '租赁方式', '车位', '用水', '用电', '采暖']], 
    drop_first=True
)

df = pd.concat([df, dummy_features], axis=1)

# 多标签二值化 (配套设施 - Point 5)
if '配套设施' in df.columns:
    amenities = ['洗衣机','暖气','空调','床','衣柜','宽带','电视','热水器','冰箱','天然气']
    for amenity in amenities:
        df[f'配套_{amenity}'] = df['配套设施'].astype(str).str.contains(amenity, na=False).astype(int)
        
# 多标签二值化 (物业类别 - Point 4)
if '物业类别' in df.columns:
    main_types_wuye = ['别墅','车库','底商','花园洋房','普通住宅','商业','商业办公类','商住两用','写字楼','公寓','住宅','单身公寓（住宅）','商务型公寓','地下仓储','工业厂房','旧式里弄','老公寓','平房','商务公寓','四合院','新式里弄','库房','酒店式公寓','公寓（住宅）','住宅式公寓','人防车位']
    for p_type in main_types_wuye:
         df[f'物业_{p_type}'] = df['物业类别'].astype(str).str.contains(p_type, na=False).astype(int)

# 多标签二值化 (建筑结构 - Point 2)
if '建筑结构' in df.columns:
    main_types_jiegou = ['塔楼', '板楼', '塔板结合', '平房']
    for j_type in main_types_jiegou:
        df[f'建筑结构_{j_type}'] = df['建筑结构'].astype(str).str.contains(j_type, na=False).astype(int)

# 多标签二值化 (产权描述 - Point 2)
if '产权描述' in df.columns:
    main_types_chanquan = ['房改房','商品房','经济适用房','集资房','使用权','私产','已购公房','拆迁还建房','二类经济适用房','售后公房','央产房','一类经济适用房','宅基房','动迁安置房','乡产','自住型商品房','限价商品房','定向安置房','公租房','安居型商品房','军产','共有产权房','廉租房','回迁房','校产']
    for p_type in main_types_chanquan:
        df[f'产权_{p_type}'] = df['产权描述'].astype(str).str.contains(p_type, na=False).astype(int)

# --- 多标签二值化 供暖 ---
if '供暖' in df.columns:
    main_types_gongnuan = ['集中供暖', '自采暖', '无供暖', '平房']
    for j_type in main_types_gongnuan:
            df[f'供暖_{j_type}'] = df['供暖'].astype(str).str.contains(j_type, na=False).astype(int)
        

# --- 多标签二值化 供暖 ---
if '供暖' in df.columns:
    main_types_gongnuan = ['集中供暖', '自采暖', '无供暖', '平房']
    for j_type in main_types_gongnuan:
            df[f'供暖_{j_type}'] = df['供暖'].astype(str).str.contains(j_type, na=False).astype(int)

if '供水' in df.columns:
    # 将混合类别拆分为独立特征
    df['供水_商水'] = df['供水'].str.contains('商水', na=False).astype(int)
    df['供水_民水'] = df['供水'].str.contains('民水', na=False).astype(int)
    # 删除原始的供水列，避免后续One-Hot编码产生混合类别
    df = df.drop('供水', axis=1)

if '供电' in df.columns:
    df['供电_商电'] = df['供电'].str.contains('商电', na=False).astype(int)
    df['供电_民电'] = df['供电'].str.contains('民电', na=False).astype(int)
    df = df.drop('供电', axis=1)

In [ ]:
# --- STAGE 6: 特征工程 ---
#交易季节性
# 提取月份特征 (1-12)
df['交易月份'] = df['交易时间'].dt.month.astype('Int64') 

# 3. 提取季节特征 (假设为北半球标准，并用数字编码)
# 1=春(Mar-May), 2=夏(Jun-Aug), 3=秋(Sep-Nov), 4=冬(Dec-Feb)

# 定义条件和对应的值
conditions = [
    # 春季：3, 4, 5 月
    (df['交易月份'].isin([3, 4, 5])),
    # 夏季：6, 7, 8 月
    (df['交易月份'].isin([6, 7, 8])),
    # 秋季：9, 10, 11 月
    (df['交易月份'].isin([9, 10, 11])),
    # 冬季：12, 1, 2 月
    (df['交易月份'].isin([12, 1, 2]))
]

# 对应季节的编码
choices = [1, 2, 3, 4] 

# 使用 np.select 根据条件赋值，default=np.nan 处理 NaT 或缺失月份
df['交易季节'] = np.select(conditions, choices, default=np.nan) 
#独热编码
df['交易月份'] = df['交易月份'].astype('category')
df['交易季节'] = df['交易季节'].astype('category')
# 使用 pd.get_dummies 进行独热编码
DUMMY_COLS = ['交易月份', '交易季节']

df_dummies = pd.get_dummies(
    df[DUMMY_COLS],
    prefix=['月', '季'],  
    dummy_na=False      
)
# 合并新特征原始列 
df = pd.concat([df, df_dummies], axis=1)

#截断处理

#3右偏截断
def clip_rightoutliers(df, col_name, upper_q=0.99):
    """
    对指定列进行分位数截断（Winsorizing）。
    Args:
        df (pd.DataFrame): DataFrame.
        col_name (str): 需要截断的列名。
        lower_q (float): 下分位数 (0.0 到 1.0)。
        upper_q (float): 上分位数 (0.0 到 1.0)。
    """
    if col_name not in df.columns:
        print(f"Warning: 列 {col_name} 不存在，跳过截断。")
        return df

    # 2. 计算上界和下界
    upper_bound = df[col_name].quantile(upper_q)
    
    # 4. 执行截断
    # inplace=True 会直接修改原始 DataFrame
    df[col_name].clip(upper=upper_bound, inplace=True)
    
    return df
    
cols_to_rightclip = ['Price','面积','房屋总数', '楼栋总数', '停车位','停车费用','燃气费','绿化率','容积率','总楼层','室','厅','卫','distance_to_center']
for col in cols_to_rightclip:
     df = clip_rightoutliers(df, col, upper_q=0.99)

#3.3 左偏截断
def clip_leftoutliers(df, col_name, lower_q=0.01):
    if col_name not in df.columns:
        print(f"Warning: 列 {col_name} 不存在，跳过截断。")
        return df

    # 2. 计算上界
    lower_bound = df[col_name].quantile(lower_q)
    
    # 4. 执行截断
    # inplace=True 会直接修改原始 DataFrame
    df[col_name].clip(lower=lower_bound, inplace=True)
    
    return df
    
cols_to_leftclip = ['建筑年代','楼层比例']
for col in cols_to_leftclip:
     df = clip_leftoutliers(df, col, lower_q=0.05)

#房龄
df['房龄'] = 2025-df['建筑年代']



In [ ]:
# --- STAGE 7: 偏态数据处理 ---
import scipy.stats as stats
import numpy as np
'''#处理偏态数据
skew_cols=[]
#对数处理
def skew_statics(df, col_name):
    """
    对指定的列进行 log1p 转换，并将结果存储在一个新列中。

    Args:
        df (pd.DataFrame): 待处理的DataFrame。
        col_name (str): 需要转换的列名（如 'Price'）
    """
    
    new_col_name = 'log_' + col_name
    df.loc[:, new_col_name] = np.log1p(df[col_name])
    df.drop(col_name,axis=1, inplace=True)

    return df

for col in skew_cols:
    df = skew_statics(df, col_name=col)
'''

#Yeo-Johnson处理
stubborn_cols = ['物业费','总楼层', 'distance_to_center','Price','面积','房屋总数','供热费','停车位','停车费用','租期_月数','绿化率','容积率','房龄'
           ,'楼栋总数']

new_cols_created = [] # 用来追踪成功创建的列
cols_to_drop = []     # 用来追踪成功处理的旧列

# 2. 循环处理
for col in stubborn_cols:
    
    valid_data = df[col].dropna()
    try:
        # 3. 核心：应用变换
        transformed_data, best_lambda = stats.yeojohnson(valid_data)
        
        # 4. 创建新列名
        new_col = f'yj_{col}' # 比如 'yj_log_Price'
        
        # 5. 把结果安全地放回原位 (只填充非 NaN 的行)
        df.loc[df[col].notna(), new_col] = transformed_data

        print(f"处理完毕: '{col}' -> '{new_col}' (最佳 Lambda: {best_lambda:.4f})")
        
        # 记录下来，准备删掉
        new_cols_created.append(new_col)
        cols_to_drop.append(col)

    except Exception as e:
        print(f"处理 '{col}' 时出错: {e}。该列可能不是数字，已跳过。")

# 6. 统一删除所有被成功处理过的旧列
df.drop(columns=cols_to_drop,axis=1,inplace=True)

处理完毕: '物业费' -> 'yj_物业费' (最佳 Lambda: -0.8040)
处理完毕: '总楼层' -> 'yj_总楼层' (最佳 Lambda: 0.5275)
处理完毕: 'distance_to_center' -> 'yj_distance_to_center' (最佳 Lambda: -0.9512)
处理完毕: 'Price' -> 'yj_Price' (最佳 Lambda: -0.1306)
处理完毕: '面积' -> 'yj_面积' (最佳 Lambda: 0.5267)
处理完毕: '房屋总数' -> 'yj_房屋总数' (最佳 Lambda: 0.2294)
处理完毕: '供热费' -> 'yj_供热费' (最佳 Lambda: -0.1212)
处理完毕: '停车位' -> 'yj_停车位' (最佳 Lambda: 0.1704)
处理完毕: '停车费用' -> 'yj_停车费用' (最佳 Lambda: 0.2665)
处理完毕: '租期_月数' -> 'yj_租期_月数' (最佳 Lambda: -1.1029)
处理完毕: '绿化率' -> 'yj_绿化率' (最佳 Lambda: 2.0752)
处理完毕: '容积率' -> 'yj_容积率' (最佳 Lambda: -0.5849)
处理完毕: '房龄' -> 'yj_房龄' (最佳 Lambda: 0.2608)
处理完毕: '楼栋总数' -> 'yj_楼栋总数' (最佳 Lambda: -0.1085)


In [ ]:
# --- STAGE 8: 删除多余数据 ---
df.drop(['城市','户型','楼层','朝向','交易时间','租赁方式','车位','用水','用电','采暖',
    '租期','配套设施','区县','板块','物业类别','建筑年代','绿 化 率','容 积 率',
    '物 业 费','建筑结构','产权描述','供暖','相对位置','楼层_数值','交易月份','交易季节','付款方式'],axis=1,inplace=True)

In [ ]:
#-----------STAGE 9 交互项构建 -----------
import pandas as pd
import numpy as np

print("--- STAGE X (新): 正在创建 11 个手动选择的特征 (用于租金模型)... ---")
print("警告: 特征 'num_pipe__yj_Price_sq' 是严重的数据泄漏。")
print("      它已被自动移除，不会被创建。")

# --- 1. 创建平方项 (共 4 项) ---

# 对应: num_pipe__yj_面积_sq
df['yj_面积_sq'] = df['yj_面积'] ** 2
# 对应: num_pipe__yj_停车费用_sq
df['yj_停车费用_sq'] = df['yj_停车费用'] ** 2
# 对应: num_pipe__yj_租期_月数_sq
df['yj_租期_月数_sq'] = df['yj_租期_月数'] ** 2
# 对应: num_pipe__卫_sq
df['卫_sq'] = df['卫'] ** 2

# --- 2. 创建交互项 (共 7 项) ---

# 对应: num_pipe__yj_物业费_x_yj_面积
df['yj_物业费_x_yj_面积'] = df['yj_物业费'] * df['yj_面积']
# 对应: num_pipe__yj_物业费_x_lat
df['yj_物业费_x_lat'] = df['yj_物业费'] * df['lat']
# 对应: num_pipe__yj_物业费_x_室
df['yj_物业费_x_室'] = df['yj_物业费'] * df['室']
# 对应: num_pipe__yj_面积_x_yj_房屋总数
df['yj_面积_x_yj_房屋总数'] = df['yj_面积'] * df['yj_房屋总数']
# 对应: num_pipe__厅_x_yj_房屋总数
df['厅_x_yj_房屋总数'] = df['厅'] * df['yj_房屋总数']
# 对应: num_pipe__yj_停车费用_x_lat
df['yj_停车费用_x_lat'] = df['yj_停车费用'] * df['lat']
# 对应: num_pipe__lat_x_室
df['lat_x_室'] = df['lat'] * df['室']

print(f"--- 手动特征创建完毕。 df 维度: {df.shape} ---")
print("您现在可以运行您原来的 K-Means + OLS 回归。")

--- STAGE X (新): 正在创建 11 个手动选择的特征 (用于租金模型)... ---
警告: 特征 'num_pipe__yj_Price_sq' 是严重的数据泄漏。
      它已被自动移除，不会被创建。
--- 手动特征创建完毕。 df 维度: (108672, 139) ---
您现在可以运行您原来的 K-Means + OLS 回归。


In [ ]:
# --- STAGE 10: ols回归+k-means超参数优化 ---
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer 
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 ---
print("--- 0. 准备数据 ---")

# 假设 'df' 是您 Cell 90 (b87bbef6) 运行到底的最终 DataFrame
# (它此时应该仍然包含 lon, lat, 但不含 '板块', '区县')
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

print(f"    ...数据已拆分: 训练集 {train_df.shape}, 测试集 {test_df.shape}")

# --- 1. (重要) 定义逆向 Yeo-Johnson 转换 ---
# (Lambda值 -0.1306 来自您 Cell 89 的输出)
PRICE_LAMBDA = -0.1306

def inverse_yeojohnson(y_yj, lambda_):
    """
    Yeo-Johnson 逆向转换 (仅适用于 y_yj >= 0, lambda != 0)
    """
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 # 规避负数
    x = np.power(val, 1/lambda_) - 1
    return x

# --- 2. 定义评估指标 (MAE) ---
def original_price_mae_scorer(y_yj, y_pred_yj):
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 3. 定义最终要进入模型的特征 ---
TARGET = 'yj_Price' # <--- 您的目标
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' # <--- OLS 模型不直接使用它们，而是使用 cluster
]

# 获取所有剩余特征
all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns # 确保测试集也有
]

# (这和 price 脚本的逻辑一样)
# 找出所有 float64 (需要 Impute + Scale)
numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

# 4. 定义目标 y
y_train = train_df[TARGET]

print(f"    ...准备完毕。最终 OLS 模型将使用 {len(all_model_features)} 个特征。")
print(f"    ...数值特征 (将被 Impute/Scale): {numeric_features_to_process[:3]} ...等 (共 {len(numeric_features_to_process)} 个)")

# --- Part A: 运行 K-Means 超参数搜索 (寻找 best_k) ---
print("\n--- Part A: 正在运行 K-Means 超参数搜索 (CV MAE)... ---")

def run_ols_with_k(k, train_data, model_features, numeric_features, y_target):
    """
    使用指定的k值运行K-Means，并返回OLS的6折交叉验证MAE
    """
    print(f"--- 正在测试 K = {k} ---")
    
    # 1. K-Means 聚类 (使用原始 lon/lat 拟合)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(train_data[['lon', 'lat']])
    
    # 2. 准备特征矩阵
    X_train_k = train_data[model_features].copy()
    X_train_k['cluster'] = kmeans.predict(train_data[['lon', 'lat']]).astype(str)
    
    # 3. 定义预处理器
    categorical_features_k = ['cluster'] # <--- 唯一需要 OHE 的新特征
    
    # 数值特征的子管道：捕获NaN -> 标准化
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')), # <-- 修复 NaN 错误
        ('scaler', StandardScaler())
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_k),
            ('num_pipe', numeric_transformer, numeric_features) # <-- 应用于所有 float64
        ],
        remainder='passthrough' # <-- 关键: 所有已编码的 bool/int32 特征将原样通过
    )
    
    # 4. 创建 OLS 管道
    pipeline_ols = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])
    
    # 5. 运行交叉验证
    start_time = time.time()
    cv_scores = cross_val_score(
        pipeline_ols, X_train_k, y_target,
        cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
    )
    end_time = time.time()
    
    mean_mae = -np.mean(cv_scores)
    print(f"K = {k}: 平均 MAE = {mean_mae:,.2f}  (耗时: {end_time - start_time:.2f} 秒)")
    
    return mean_mae

# 运行循环 (使用一个合理的步长进行搜索)
k_range = range(200,201,10) 
print(f"--- (注意) 正在测试 K 范围: {list(k_range)} ---")

results = {}

for k in k_range:
    results[k] = run_ols_with_k(
        k=k,
        train_data=train_df, 
        model_features=all_model_features,
        numeric_features=numeric_features_to_process, # <-- 使用修正的列表
        y_target=y_train
    )

# --- Part B: 使用 best_k 评估最终模型 ---
print("\n--- Part B: 正在使用 Best K 评估最终 OLS 模型 ---")

# 1. 找出最佳 K 和对应的 CV MAE
best_k = min(results, key=results.get)
mae_cv = results[best_k]
print(f"    ...K-Means 优化完成。最佳 K = {best_k} (CV MAE: {mae_cv:,.2f})")

# 2. 创建最终的 K-Means
print("    ...正在构建最终管道...")
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 3. 准备最终的 X_train (带 'cluster' 特征)
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

# 4. 定义最终的 OLS 管道
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process) # <-- 使用修正的列表
    ],
    remainder='passthrough'
)
final_pipeline_ols = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', LinearRegression())
])

# 5. 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_ols.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_ols.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6. 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_ols.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_ols.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- Part C: 报告所有 OLS 关键评估指标 (包含您要的输出) ---
print("\n" + "="*50)
print("--- OLS 最终评估指标 ('rent' data, MAE) ---")
print("="*50)

print("\nK-Means 超参数搜索结果 (MAE vs. K):")
print_results = pd.DataFrame.from_dict(results, orient='index', columns=['MAE'])
print_results.index.name = 'K (聚类数)'
print(print_results)

print("\n最终模型性能:")
print(f"最佳 K-Means 聚类数: {best_k}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

--- 0. 准备数据 ---
    ...数据已拆分: 训练集 (98899, 139), 测试集 (9773, 139)
    ...准备完毕。最终 OLS 模型将使用 134 个特征。
    ...数值特征 (将被 Impute/Scale): ['电梯', '燃气', '燃气费'] ...等 (共 31 个)

--- Part A: 正在运行 K-Means 超参数搜索 (CV MAE)... ---
--- (注意) 正在测试 K 范围: [200] ---
--- 正在测试 K = 200 ---
K = 200: 平均 MAE = 98,736.51  (耗时: 14.20 秒)

--- Part B: 正在使用 Best K 评估最终 OLS 模型 ---
    ...K-Means 优化完成。最佳 K = 200 (CV MAE: 98,736.51)
    ...正在构建最终管道...
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- OLS 最终评估指标 ('rent' data, MAE) ---

K-Means 超参数搜索结果 (MAE vs. K):
                  MAE
K (聚类数)              
200      98736.509344

最终模型性能:
最佳 K-Means 聚类数: 200
In-sample MAE (训练集):      98,329.64
Out-of-sample MAE (20%验证集): 99,620.57
Cross-validation MAE (6-fold): 98,736.51


In [ ]:
# --- 生成测试集预测 ---
print("\n--- Part D: 正在生成测试集预测... ---")
 
# 准备 X_test
X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)
 
# (final_pipeline_ols 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_ols.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_rent_ols.csv', index=False)
print("已生成对测试集的预测文件 'prediction_rent_ols.csv'")


--- Part D: 正在生成测试集预测... ---
已生成对测试集的预测文件 'prediction_rent_ols.csv'


In [ ]:
# --- STAGE 11: lasso回归+k-means超参数优化 ---
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LassoCV # <--- (新) 导入 LassoCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer 
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 (与 OLS 单元格相同) ---
print("--- 0. 准备数据 (用于 Lasso)... ---")
# (我们使用的是 Cell 36 运行后内存中的 'df')
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 1. 定义逆向转换和评估器 (与 OLS 单元格相同) ---
# (Lambda值 -0.1306 来自您 Cell 34 的输出)
PRICE_LAMBDA = -0.1306

def inverse_yeojohnson(y_yj, lambda_):
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 # 规避负数
    x = np.power(val, 1/lambda_) - 1
    return x

def original_price_mae_scorer(y_yj, y_pred_yj):
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 2. 定义特征 (与 OLS 单元格相同) ---
TARGET = 'yj_Price'
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' 
]

all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns
]

numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

y_train = train_df[TARGET]

print(f"    ...准备完毕。Lasso 模型将使用 {len(all_model_features)} 个特征。")

# --- 3. K-Means 特征工程 (使用 OLS 找到的最佳 K=200) ---
print(f"--- 3. 正在应用 K-Means (K=200)... ---")
best_k = 200 # <-- 来自 OLS 单元格 的结果
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 准备 X_train 和 X_test
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)

# --- 4. 定义预处理器 (与 OLS 单元格相同) ---
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process)
    ],
    remainder='passthrough'
)

# --- 5. (新) 定义 LassoCV 回归管道 ---
print("--- 5. 正在构建 LassoCV 管道... ---")

# LassoCV 会自动使用 6 折交叉验证 (cv=kf_6) 寻找最佳的 alpha (正则化强度)
lasso_model = LassoCV(cv=kf_6, n_jobs=-1, random_state=42)

# 组装最终管道
final_pipeline_lasso = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', lasso_model) # <-- (新) 将回归器替换为 LassoCV
])

# --- 6. (新) 评估 Lasso 模型 ---

# 6.1 计算 Cross-validation MAE
print("    ...正在计算 Cross-validation MAE (6-fold)...")
start_time_cv = time.time()
cv_scores = cross_val_score(
    final_pipeline_lasso, X_train_final, y_train,
    cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
)
mae_cv = -np.mean(cv_scores)
print(f"    (CV 耗时: {time.time() - start_time_cv:.2f} 秒)")

# 6.2 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_lasso.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_lasso.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6.3 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_lasso.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_lasso.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- 7. (新) 报告 Lasso 评估指标 ---
print("\n" + "="*50)
print("--- LassoCV 最终评估指标 ('rent' data, MAE) ---")
print("="*50)
print(f"最佳 K-Means 聚类数: {best_k}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

# --- 8. (新) 生成 Lasso 预测文件 ---
print("\n--- Part D: 正在生成 Lasso 测试集预测... ---")

# (final_pipeline_lasso 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_lasso.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_rent_lasso.csv', index=False) # <-- (新) 命名
print("已生成对测试集的预测文件 'prediction_rent_lasso.csv'")

--- 0. 准备数据 (用于 Lasso)... ---
    ...准备完毕。Lasso 模型将使用 134 个特征。
--- 3. 正在应用 K-Means (K=200)... ---
--- 5. 正在构建 LassoCV 管道... ---
    ...正在计算 Cross-validation MAE (6-fold)...
    (CV 耗时: 83.03 秒)
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- LassoCV 最终评估指标 ('rent' data, MAE) ---
最佳 K-Means 聚类数: 200
In-sample MAE (训练集):      117,105.31
Out-of-sample MAE (20%验证集): 118,989.18
Cross-validation MAE (6-fold): 117,442.84

--- Part D: 正在生成 Lasso 测试集预测... ---
已生成对测试集的预测文件 'prediction_rent_lasso.csv'


In [ ]:
# 输出alpha
all_alphas_tested = final_pipeline_lasso.named_steps['regressor'].alphas_
print(f"\nLassoCV 测试过的 Alpha 路径 (共 {len(all_alphas_tested)} 个):")
print(all_alphas_tested)


LassoCV 测试过的 Alpha 路径 (共 100 个):
[6.17886926e-02 5.76243415e-02 5.37406538e-02 5.01187136e-02
 4.67408800e-02 4.35907012e-02 4.06528338e-02 3.79129689e-02
 3.53577616e-02 3.29747668e-02 3.07523779e-02 2.86797706e-02
 2.67468500e-02 2.49442019e-02 2.32630461e-02 2.16951947e-02
 2.02330112e-02 1.88693739e-02 1.75976413e-02 1.64116192e-02
 1.53055310e-02 1.42739894e-02 1.33119703e-02 1.24147880e-02
 1.15780729e-02 1.07977495e-02 1.00700173e-02 9.39133187e-03
 8.75838754e-03 8.16810153e-03 7.61759882e-03 7.10419816e-03
 6.62539898e-03 6.17886926e-03 5.76243415e-03 5.37406538e-03
 5.01187136e-03 4.67408800e-03 4.35907012e-03 4.06528338e-03
 3.79129689e-03 3.53577616e-03 3.29747668e-03 3.07523779e-03
 2.86797706e-03 2.67468500e-03 2.49442019e-03 2.32630461e-03
 2.16951947e-03 2.02330112e-03 1.88693739e-03 1.75976413e-03
 1.64116192e-03 1.53055310e-03 1.42739894e-03 1.33119703e-03
 1.24147880e-03 1.15780729e-03 1.07977495e-03 1.00700173e-03
 9.39133187e-04 8.75838754e-04 8.16810153e-04 7.617

In [ ]:
# --- STAGE 12: ridge回归+k-means超参数优化 ---
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import RidgeCV # <--- (新) 导入 RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer 
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 (与 OLS/Lasso 单元格相同) ---
print("--- 0. 准备数据 (用于 Ridge)... ---")
# (我们使用的是 Cell 36 运行后内存中的 'df')
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 1. 定义逆向转换和评估器 (与 OLS/Lasso 单元格相同) ---
# (Lambda值 -0.1306 来自您 Cell 34 的输出)
PRICE_LAMBDA = -0.1306

def inverse_yeojohnson(y_yj, lambda_):
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 # 规避负数
    x = np.power(val, 1/lambda_) - 1
    return x

def original_price_mae_scorer(y_yj, y_pred_yj):
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 2. 定义特征 (与 OLS/Lasso 单元格相同) ---
TARGET = 'yj_Price'
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' 
]

all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns
]

numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

y_train = train_df[TARGET]

print(f"    ...准备完毕。Ridge 模型将使用 {len(all_model_features)} 个特征。")

# --- 3. K-Means 特征工程 (使用 OLS 找到的最佳 K=200) ---
print(f"--- 3. 正在应用 K-Means (K=200)... ---")
best_k = 200 # <-- 来自 OLS 单元格 的结果
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 准备 X_train 和 X_test
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)

# --- 4. 定义预处理器 (与 OLS/Lasso 单元格相同) ---
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process)
    ],
    remainder='passthrough'
)

# --- 5. (新) 定义 RidgeCV 回归管道 ---
print("--- 5. 正在构建 RidgeCV 管道... ---")

# (新) RidgeCV 通常需要我们提供一个 alpha 列表进行测试
# 这是一个对数间隔的列表 (例如: 0.1, 1.0, 10.0, 100.0)
alphas_to_test = np.logspace(-2, 4, 10) 
print(f"    ...将测试 {len(alphas_to_test)} 个 Alpha 值 (从 {alphas_to_test[0]} 到 {alphas_to_test[-1]})")

# RidgeCV 会自动使用 6 折交叉验证 (cv=kf_6) 从列表中寻找最佳 alpha
ridge_model = RidgeCV(alphas=alphas_to_test, cv=kf_6)

# 组装最终管道
final_pipeline_ridge = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', ridge_model) # <-- (新) 将回归器替换为 RidgeCV
])

# --- 6. (新) 评估 Ridge 模型 ---

# 6.1 计算 Cross-validation MAE
# (注意: 这里我们再次对 *整个* 管道进行 CV，以获得最终的性能分数)
print("    ...正在计算 Cross-validation MAE (6-fold)...")
start_time_cv = time.time()
cv_scores = cross_val_score(
    final_pipeline_ridge, X_train_final, y_train,
    cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
)
mae_cv = -np.mean(cv_scores)
print(f"    (CV 耗时: {time.time() - start_time_cv:.2f} 秒)")

# 6.2 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_ridge.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_ridge.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6.3 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_ridge.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_ridge.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- 7. (新) 报告 Ridge 评估指标 ---
print("\n" + "="*50)
print("--- RidgeCV 最终评估指标 ('rent' data, MAE) ---")
print("="*50)
print(f"最佳 K-Means 聚类数: {best_k}")
# (新) 打印 Ridge 找到的最佳 alpha
best_alpha_ridge = final_pipeline_ridge.named_steps['regressor'].alpha_
print(f"Ridge 找到的最佳 Alpha: {best_alpha_ridge}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

# --- 8. (新) 生成 Ridge 预测文件 ---
print("\n--- Part D: 正在生成 Ridge 测试集预测... ---")

# (final_pipeline_ridge 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_ridge.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_rent_ridge.csv', index=False) # <-- (新) 命名
print("已生成对测试集的预测文件 'prediction_rent_ridge.csv'")

--- 0. 准备数据 (用于 Ridge)... ---
    ...准备完毕。Ridge 模型将使用 134 个特征。
--- 3. 正在应用 K-Means (K=200)... ---
--- 5. 正在构建 RidgeCV 管道... ---
    ...将测试 10 个 Alpha 值 (从 0.01 到 10000.0)
    ...正在计算 Cross-validation MAE (6-fold)...
    (CV 耗时: 3003.94 秒)
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- RidgeCV 最终评估指标 ('rent' data, MAE) ---
最佳 K-Means 聚类数: 200
Ridge 找到的最佳 Alpha: 0.046415888336127774
In-sample MAE (训练集):      98,329.21
Out-of-sample MAE (20%验证集): 99,620.68
Cross-validation MAE (6-fold): 98,736.01

--- Part D: 正在生成 Ridge 测试集预测... ---
已生成对测试集的预测文件 'prediction_rent_ridge.csv'


*Price_model*

In [ ]:
#-----------STAGE 1 数据加载 & 单位转换------------
import pandas as pd
import numpy  as np
import re
#1.加载数据
file_path = 'comment_replaced__price.csv'
df = pd.read_csv(file_path,encoding='gbk')

#2.删除不需要的列
df.drop(['环线','抵押信息','coord_x','coord_y','物业办公电话','区县','板块_comm','物业公司','配备电梯','别墅类型','环线位置','开发商','产权描述','房屋年限','年份'],axis=1,inplace=True)

# 3. 数据预处理

# 3.1 建筑面积、套内面积、户数、楼栋总数转化
df['建筑面积'] = df['建筑面积'].str.replace('㎡', '', regex=False).astype(float)
df['套内面积'] = df['套内面积'].str.replace('㎡', '', regex=False).astype(float)
df['房屋总数'] = df['房屋总数'].str.replace('户', '', regex=False).astype(float)
df['楼栋总数'] = df['楼栋总数'].str.replace('栋', '', regex=False).astype(float)
df['燃气费'] = df['燃气费'].str.replace('元/m³','', regex=False)
df['供热费'] = df['供热费'].str.replace('元/㎡','', regex=False)

# 3.2 交易时间和上次交易转化成时间
df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')
df['上次交易'] = pd.to_datetime(df['上次交易'], errors='coerce')

# 3.3 建筑年代转化成均值
def parse_build_year(s):
    if pd.isnull(s): return np.nan
    s = str(s); years = re.findall(r'(\d{4})', s)
    if not years: return np.nan
    return np.mean([float(y) for y in years]) if len(years) > 1 else float(years[0])
df['建筑年代'] = df['建筑年代'].apply(parse_build_year)

# 3.4 绿化率百分比字符串（如 '35%'）转换为数值比例（如 0.35）。
df['绿化率'] = pd.to_numeric(df['绿 化 率'].astype(str).str.replace('%', '', regex=False), errors='coerce') / 100.0

# 3.5 物业费转化成均值
def parse_range_mean(s):
    if pd.isnull(s):
        return np.nan

    s = str(s).strip() # 转换为字符串并移除空格，使用正则表达式查找所有浮点数或整数：\d+ 匹配一个或多个数字 \.? 匹配零个或一个点 \d* 匹配零个或多个数字（在点后面）
    numbers = re.findall(r'(\d+\.?\d*)', s)

    if not numbers:
        return np.nan
    float_numbers = [float(n) for n in numbers]

    # 如果找到多个数字，返回它们的平均值；否则返回找到的唯一值
    if len(float_numbers) > 1:
        return np.mean(float_numbers)
    else:
        return float_numbers[0]
    
df['物业费'] = df['物 业 费'].apply(parse_range_mean)
df['燃气费'] = df['燃气费'].apply(parse_range_mean)
df['供热费'] = df['供热费'].apply(parse_range_mean)
df['容积率'] = df['容 积 率']

def get_first_number(s):
    if pd.isnull(s):
        return np.nan

    s = str(s)
    match = re.search(r'(\d+\.?\d*)', s)

    if match:
        return float(match.group(0))
    else:
        return np.nan
df['停车费用'] = df['停车费用'].apply(get_first_number)

#计算距离
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    计算两点之间的 Haversine 距离 (单位：公里)
    """
    R = 6371  # 地球半径 (公里)
    
    # 确保输入是浮点数并处理潜在的NaN
    try:
        lon1, lat1, lon2, lat2 = map(float, [lon1, lat1, lon2, lat2])
    except (ValueError, TypeError):
        return np.nan # 如果任何一个是NaN或无法转换，返回NaN
        
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    distance = R * c
    return distance

C:\Users\12611\AppData\Local\Temp\ipykernel_1072\2439291030.py:9: DtypeWarning: Columns (4,33,47,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path,encoding='gbk')


In [ ]:
# --- STAGE 2 : 拆分、计算规则、填充、计算距离、合并 ---

# (辅助函数：安全地获取众数)
def _get_mode(series):
    mode = series.mode()
    return mode.iloc[0] if not mode.empty else np.nan

# (辅助函数：应用规则)
def _apply_fill_rules(df, rules):
    df_processed = df.copy()
    
    # 1. 填充数值型列
    num_cols = rules.get('numerical_cols', [])
    for col in num_cols:
        if '板块' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['板块'].map(rules['sector_medians'].get(col, {})))
        if '区域' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['区域'].map(rules['region_medians'].get(col, {})))
        if '城市' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['城市'].map(rules['city_medians'].get(col, {})))
        df_processed[col] = df_processed[col].fillna(rules.get('global_medians', {}).get(col))
    
    # 2. 填充对象型列
    cat_cols = rules.get('categorical_cols', [])
    for col in cat_cols:
        if '板块' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['板块'].map(rules['sector_modes'].get(col, {})))
        if '区域' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['区域'].map(rules['region_modes'].get(col, {})))
        if '城市' in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed['城市'].map(rules['city_modes'].get(col, {})))
        df_processed[col] = df_processed[col].fillna(rules.get('global_modes', {}).get(col))
                
    # 3. 填充 '上次交易'
    if '上次交易' in df_processed.columns:
        first_reg_date = df_processed['建筑年代'].apply(
            lambda x: pd.to_datetime(str(int(x)) + '-01-01', errors='coerce') if pd.notna(x) else pd.NaT
        )
        df_processed['上次交易'] = df_processed['上次交易'].fillna(first_reg_date)
        df_processed['上次交易'] = df_processed['上次交易'].fillna(rules.get('fallback_registration_date'))
    
    # 4. 剪裁日期
    end_date = pd.to_datetime('2025-10-15')
    if '交易时间' in df_processed.columns: df_processed['交易时间'] = df_processed['交易时间'].clip(upper=end_date)
    if '上次交易' in df_processed.columns: df_processed['上次交易'] = df_processed['上次交易'].clip(upper=end_date)
        
    return df_processed

# --- 1. 拆分数据 ---
print("--- 1. 正在拆分数据... ---")
# 假设 'df' 是 Cell 23 的输出
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 2. 填充分组键 (板块/区域/城市) ---
print("--- 2. 正在填充分组键 (板块/区域/城市)... ---")
group_keys = ['板块', '区域', '城市']
rules_group_keys = {}
for col in group_keys:
    if col in train_df.columns:
        mode_val = train_df[col].mode()[0]
        rules_group_keys[col] = mode_val # 存储规则
        train_df[col] = train_df[col].fillna(mode_val)
        test_df[col] = test_df[col].fillna(mode_val) 

# --- 3. 计算所有填充规则 (仅在 train_df 上) ---
print("--- 3. 正在 train_df 上计算所有填充规则... ---")
train_rules = {}

# 数值型规则
numerical_cols = ['套内面积','建筑年代','房屋总数','楼栋总数','容积率','燃气费','供热费','停车费用','绿化率','物业费','停车位']
numerical_cols = [col for col in numerical_cols if col in train_df.columns]
train_rules['numerical_cols'] = numerical_cols
train_rules['global_medians'] = train_df[numerical_cols].median()
train_rules['sector_medians'] = train_df.groupby('板块')[numerical_cols].median()
train_rules['region_medians'] = train_df.groupby('区域')[numerical_cols].median()
train_rules['city_medians'] = train_df.groupby('城市')[numerical_cols].median()

# 对象型规则
categorical_cols = ['梯户比例','房屋户型','房屋朝向','建筑结构','装修情况','房屋用途','物业类别','供水','供暖','供电']
categorical_cols = [col for col in categorical_cols if col in train_df.columns]
train_rules['categorical_cols'] = categorical_cols
train_rules['global_modes'] = train_df[categorical_cols].apply(_get_mode)
train_rules['sector_modes'] = train_df.groupby('板块')[categorical_cols].apply(_get_mode)
train_rules['region_modes'] = train_df.groupby('区域')[categorical_cols].apply(_get_mode)
train_rules['city_modes'] = train_df.groupby('城市')[categorical_cols].apply(_get_mode)

# '上次交易' 规则
median_build_year = train_df['建筑年代'].median()
train_rules['fallback_registration_date'] = pd.to_datetime(
    str(int(median_build_year)) + '-01-01', errors='coerce'
) if pd.notna(median_build_year) else pd.NaT

print("    ...规则计算完毕。")

# --- 4. 应用填充规则 ---
print("--- 4. 正在应用填充规则到 train_df 和 test_df... ---")
train_df_filled = _apply_fill_rules(train_df, train_rules)
test_df_filled = _apply_fill_rules(test_df, train_rules)

# --- 5. 计算并应用 Distance 特征 ---
print("--- 5. 正在计算和应用 'distance_to_center' 特征... ---")

# 'Fit': 只在训练集上计算中心
sector_centers = train_df_filled.groupby('板块')[['lon', 'lat']].mean().reset_index()
sector_centers.rename(columns={'lon': 'sector_center_lon', 'lat': 'sector_center_lat'}, inplace=True)
global_center_lon = train_df_filled['lon'].mean()
global_center_lat = train_df_filled['lat'].mean()

# 'Transform': 应用到 train_df_filled
train_df_filled = pd.merge(train_df_filled, sector_centers, on='板块', how='left')
train_df_filled['distance_to_center'] = train_df_filled.apply(
    lambda row: haversine_distance(row['lon'], row['lat'], row['sector_center_lon'], row['sector_center_lat']), 
    axis=1
)

# 'Transform': 应用到 test_df_filled
test_df_filled = pd.merge(test_df_filled, sector_centers, on='板块', how='left')
test_df_filled['sector_center_lon'].fillna(global_center_lon, inplace=True)
test_df_filled['sector_center_lat'].fillna(global_center_lat, inplace=True)
test_df_filled['distance_to_center'] = test_df_filled.apply(
    lambda row: haversine_distance(row['lon'], row['lat'], row['sector_center_lon'], row['sector_center_lat']), 
    axis=1
)

# 清理辅助列
train_df_filled.drop(['sector_center_lon', 'sector_center_lat'], axis=1, inplace=True)
test_df_filled.drop(['sector_center_lon', 'sector_center_lat'], axis=1, inplace=True)
print("    ...'distance_to_center' 特征已创建。")

# --- 6. 重新合并 ---
print("--- 6. 正在重新合并数据以便后续处理... ---")
df = pd.concat([train_df_filled, test_df_filled], ignore_index=True)

print(f"\n--- 流程完成。新的 'df' 维度: {df.shape} ---")
print("您现在可以继续运行 Cell 25 (STAGE 3 特征工程)。")

--- 1. 正在拆分数据... ---
--- 2. 正在填充分组键 (板块/区域/城市)... ---
--- 3. 正在 train_df 上计算所有填充规则... ---
    ...规则计算完毕。
--- 4. 正在应用填充规则到 train_df 和 test_df... ---
--- 5. 正在计算和应用 'distance_to_center' 特征... ---


C:\Users\12611\AppData\Local\Temp\ipykernel_1072\117656104.py:119: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df_filled['sector_center_lon'].fillna(global_center_lon, inplace=True)
C:\Users\12611\AppData\Local\Temp\ipykernel_1072\117656104.py:120: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always 

    ...'distance_to_center' 特征已创建。
--- 6. 正在重新合并数据以便后续处理... ---

--- 流程完成。新的 'df' 维度: (137888, 46) ---
您现在可以继续运行 Cell 25 (STAGE 3 特征工程)。


In [ ]:
#-----------STAGE 3 部分特征工程------------
#房屋户型处理
if '房屋户型' in df.columns:
    # 使用 .str.extract 并指定 expand=False 返回 Series，然后转换类型和填充
    df['室'] = df['房屋户型'].str.extract(r'(\d+)室', expand=False).astype(float)
    df['厅'] = df['房屋户型'].str.extract(r'(\d+)厅', expand=False).astype(float)
    df['卫'] = df['房屋户型'].str.extract(r'(\d+)卫', expand=False).astype(float)
    df['厨'] = df['房屋户型'].str.extract(r'(\d+)厨', expand=False).astype(float)
    df.loc[df['房屋户型'].str.contains('房间', na=False), '室'] = df['房屋户型'].str.extract(r'(\d+)房间', expand=False).astype(float)

# 所在楼层 
if '所在楼层' in df.columns:
    floor_str = df['所在楼层'].astype(str) 
    df['总楼层_temp'] = floor_str.str.extract(r'共(\d+)层')[0].astype(float)
    floor_position_str = floor_str.str.extract(r'(地下室|底层|顶层|低楼层|中楼层|高楼层)')[0]
    floor_position_map = {'地下室': -1, '底层': 1, '低楼层': 2, '中楼层': 3, '高楼层': 4, '顶层': 5}
    df['楼层位置_mapped'] = floor_position_str.map(floor_position_map)

# 房屋朝向
if '房屋朝向' in df.columns:
    df['朝南'] = df['房屋朝向'].str.contains('南', na=False).astype(int)
    df['朝北'] = df['房屋朝向'].str.contains('北', na=False).astype(int)
    df['朝东'] = df['房屋朝向'].str.contains('东', na=False).astype(int)
    df['朝西'] = df['房屋朝向'].str.contains('西', na=False).astype(int)

# 梯户比例 
if '梯户比例' in df.columns:
    chinese_num_map = { "一": "1", "二": "2", "两": "2", "三": "3", "四": "4", "五": "5", "六": "6", "七": "7", "八": "8", "九": "9", "十": "10", "十一": "11", "十二": "12", "十三": "13", "十四": "14", "十五": "15", "十六": "16", "十七": "17", "十八": "18", "十九": "19", "二十": "20", "二十一": "21", "二十二": "22", "二十三": "23", "二十四": "24", "二十五": "25", "二十六": "26", "二十七": "27", "二十八": "28", "二十九": "29", "三十": "30", "三十一": "31", "三十二": "32", "三十三": "33", "三十四": "34", "三十五": "35", "三十六": "36", "三十七": "37", "三十八": "38", "三十九": "39", "四十": "40", "四十一": "41", "四十二": "42", "四十三": "43", "四十四": "44", "四十五": "45", "四十六": "46", "四十七": "47", "四十八": "48", "四十九": "49", "五十": "50", "五十一": "51", "五十二": "52", "五十三": "53", "五十四": "54", "五十五": "55", "五十六": "56", "五十七": "57", "五十八": "58", "五十九": "59", "六十": "60", "六十一": "61", "六十二": "62", "六十三": "63", "六十四": "64", "六十五": "65", "六十六": "66", "六十七": "67", "六十八": "68", "六十九": "69", "七十": "70" }
    temp_col = df['梯户比例'].astype(str)
    sorted_keys = sorted(chinese_num_map.keys(), key=len, reverse=True)
    for char in sorted_keys: num_str = chinese_num_map[char]; temp_col = temp_col.str.replace(char, num_str)
    df['梯'] = temp_col.str.extract(r'(\d+)梯').astype(float)
    df['户'] = temp_col.str.extract(r'(\d+)户').astype(float)
    df['梯户比'] = df['梯']/df['户']

# 供水供电
if '供水' in df.columns:
    # 将混合类别拆分为独立特征
    df['供水_商水'] = df['供水'].str.contains('商水', na=False).astype(int)
    df['供水_民水'] = df['供水'].str.contains('民水', na=False).astype(int)

if '供电' in df.columns:
    df['供电_商电'] = df['供电'].str.contains('商电', na=False).astype(int)
    df['供电_民电'] = df['供电'].str.contains('民电', na=False).astype(int)

# --- 特征工程: 映射 装修情况 ---
if '装修情况' in df.columns:
    df['装修情况'] = df['装修情况'].str.strip()
    map_zhuangxiu = {'毛坯': 0, '简装': 1, '精装': 2, '其他': 1}
    df['装修情况'] = df['装修情况'].map(map_zhuangxiu)

#特征工程
# --- 多标签二值化 (建筑结构_comm) ---
if '建筑结构_comm' in df.columns:
    main_types_jiegou_comm = ['塔楼', '板楼', '塔板结合', '平房']
    for j_type in main_types_jiegou_comm:
            df[f'建筑结构_comm_{j_type}'] = df['建筑结构_comm'].astype(str).str.contains(j_type, na=False).astype(int)

# --- 多标签二值化 (建筑结构) ---
if '建筑结构' in df.columns:
    main_types_jianzhu = ['钢混结构','混合结构','砖混结构','砖木结构','未知结构','钢结构','框架结构']
    for j_type in main_types_jianzhu:
            df[f'建筑结构_{j_type}'] = df['建筑结构'].astype(str).str.contains(j_type, na=False).astype(int)
        
            
# --- 特征工程: 多标签 (物业类别) ---
if '物业类别' in df.columns:
    main_types = ['普通住宅', '别墅', '写字楼', '商业', '公寓', '底商', '车库', '花园洋房', '平房', '新式里弄', '老公寓']
    for prop_type in main_types:
        df[f'物业类型_{prop_type}'] = df['物业类别'].astype(str).str.contains(prop_type, na=False).astype(int)
    
# --- 特征工程: 多标签 (房屋用途) ---
if '房屋用途' in df.columns:
    main_types_yongtu = [
                '普通住宅', '别墅', '商业办公类', '车库', '公寓', '酒店式公寓', 
                '四合院', '商务型公寓', '住宅式公寓', '商住两用', '新式里弄', 
                '老公寓', '花园洋房', '底商', '商业', '商务公寓', '写字楼', '住宅']
    for p_type in main_types_yongtu:
        df[f'房屋用途_{p_type}'] = df['房屋用途'].astype(str).str.contains(p_type, na=False).astype(int)

# 多标签二值化 (交易权属)
if '交易权属' in df.columns:
    main_types_chanquan = ['私产','商品房','已购公房','央产房','二类经济适用房','一类经济适用房','定向安置房','限价商品房','自住型商品房'
 '房改房','拆迁还建房','集资房','经济适用房','动迁安置房','售后公房','回迁房']
    for p_type in main_types_chanquan:
        df[f'交易权属_{p_type}'] = df['交易权属'].astype(str).str.contains(p_type, na=False).astype(int)

# 多标签二值化 (供暖 )
if '供暖' in df.columns:
    main_types_gongnuan = ['集中供暖','自采暖','无供暖']
    for p_type in main_types_gongnuan:
        df[f'供暖_{p_type}'] = df['供暖'].astype(str).str.contains(p_type, na=False).astype(int)

#产权所属0-1
df['产权所属'] = df['产权所属'].replace({'非共有': 1, '共有': 0})

#房龄
df['房龄'] = 2025-df['建筑年代']

df['持有年限'] =((df['交易时间'] - df['上次交易']).dt.days)/365.25
df['持有年限'] = df['持有年限'].clip(lower=0)

# 2. 提取月份特征 (1-12)
df['交易月份'] = df['交易时间'].dt.month.astype('Int64') 

# 3. 提取季节特征 (假设为北半球标准，并用数字编码)
# 1=春(Mar-May), 2=夏(Jun-Aug), 3=秋(Sep-Nov), 4=冬(Dec-Feb)

# 定义条件和对应的值
conditions = [
    # 春季：3, 4, 5 月
    (df['交易月份'].isin([3, 4, 5])),
    # 夏季：6, 7, 8 月
    (df['交易月份'].isin([6, 7, 8])),
    # 秋季：9, 10, 11 月
    (df['交易月份'].isin([9, 10, 11])),
    # 冬季：12, 1, 2 月
    (df['交易月份'].isin([12, 1, 2]))
]

# 对应季节的编码
choices = [1, 2, 3, 4] 

# 使用 np.select 根据条件赋值，default=np.nan 处理 NaT 或缺失月份
df['交易季节'] = np.select(conditions, choices, default=np.nan) 
#独热编码
df['交易月份'] = df['交易月份'].astype('category')
df['交易季节'] = df['交易季节'].astype('category')
# 使用 pd.get_dummies 进行独热编码
DUMMY_COLS = ['交易月份', '交易季节']

df_dummies = pd.get_dummies(
    df[DUMMY_COLS],
    prefix=['月', '季'],  
    dummy_na=False      
)
# --- 4. 合并新特征原始列 ---
df = pd.concat([df, df_dummies], axis=1)

C:\Users\12611\AppData\Local\Temp\ipykernel_1072\551773711.py:95: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['产权所属'] = df['产权所属'].replace({'非共有': 1, '共有': 0})
C:\Users\12611\AppData\Local\Temp\ipykernel_1072\551773711.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['持有年限'] =((df['交易时间'] - df['上次交易']).dt.days)/365.25
C:\Users\12611\AppData\Local\Temp\ipykernel_1072\551773711.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

In [ ]:
#-----------STAGE 4 新特征工程------------
#1. 计算得房率
df['得房率']=df['套内面积']/df['建筑面积']

#2. 文本转化
text_cols = ['周边配套', '交通出行', '核心卖点']
for col in text_cols:
    if col in df.columns:
        df[f'有_{col}'] = df[col].notnull().astype(int)
        if col == '核心卖点':
             df[f'{col}_长度'] = df[col].str.len().fillna(0)

#3. 数据截断处理
#3.2 右偏截断
def clip_rightoutliers(df, col_name, upper_q=0.99):
    """
    对指定列进行分位数截断（Winsorizing）。
    Args:
        df (pd.DataFrame): DataFrame.
        col_name (str): 需要截断的列名。
        lower_q (float): 下分位数 (0.0 到 1.0)。
        upper_q (float): 上分位数 (0.0 到 1.0)。
    """
    if col_name not in df.columns:
        print(f"Warning: 列 {col_name} 不存在，跳过截断。")
        return df

    # 2. 计算上界和下界
    upper_bound = df[col_name].quantile(upper_q)
    
    # 4. 执行截断
    # inplace=True 会直接修改原始 DataFrame
    df[col_name].clip(upper=upper_bound, inplace=True)
    
    return df
    
cols_to_rightclip = ['绿化率','容积率','室', '厅', '卫','总楼层_temp','梯','户','梯户比','房龄','持有年限','得房率','楼栋总数']
for col in cols_to_rightclip:
     df = clip_rightoutliers(df, col, upper_q=0.99)

#3.3 左偏截断
def clip_leftoutliers(df, col_name, lower_q=0.01):
    if col_name not in df.columns:
        print(f"Warning: 列 {col_name} 不存在，跳过截断。")
        return df

    # 2. 计算上界
    lower_bound = df[col_name].quantile(lower_q)
    
    # 4. 执行截断
    # inplace=True 会直接修改原始 DataFrame
    df[col_name].clip(lower=lower_bound, inplace=True)
    
    return df
    
cols_to_leftclip = ['建筑年代','厨']
for col in cols_to_leftclip:
     df = clip_leftoutliers(df, col, lower_q=0.05)

C:\Users\12611\AppData\Local\Temp\ipykernel_1072\1219588101.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col_name].clip(upper=upper_bound, inplace=True)
C:\Users\12611\AppData\Local\Temp\ipykernel_1072\1219588101.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [ ]:
#-----------STAGE 5 偏态&截断处理------------
import scipy.stats as stats
import numpy as np

#处理偏态数据
'''skew_cols=['Price','建筑面积','套内面积','房屋总数','楼栋总数','停车位','停车费用','物业费','容积率','房龄','持有年限','厨','户','卫','梯','梯户比','得房率']
def skew_statics(df, col_name):
    """
    对指定的列进行 log1p 转换，并将结果存储在一个新列中。

    Args:
        df (pd.DataFrame): 待处理的DataFrame。
        col_name (str): 需要转换的列名（如 'Price'）。
    """
    
    new_col_name = 'log_' + col_name
    df.loc[:, new_col_name] = np.log1p(df[col_name])
    df.drop(col_name,axis=1, inplace=True)

    return df

for col in skew_cols:
    df = skew_statics(df, col_name=col)
'''


stubborn_cols = ['distance_to_center','绿化率','燃气费','供热费','客户反馈',
                 'Price','建筑面积','套内面积','房屋总数','楼栋总数','停车位',
                 '停车费用','物业费','容积率','房龄','持有年限','厨','室','厅','户','卫',
                 '梯','梯户比','得房率','总楼层_temp','楼层位置_mapped']

new_cols_created = [] # 用来追踪成功创建的列
cols_to_drop = []     # 用来追踪成功处理的旧列

# 2. 循环处理
for col in stubborn_cols:
    
    valid_data = df[col].dropna()
    try:
        # 3. 核心：应用变换
        transformed_data, best_lambda = stats.yeojohnson(valid_data)
        
        # 4. 创建新列名
        new_col = f'yj_{col}' # 比如 'yj_log_Price'
        
        # 5. 把结果安全地放回原位 (只填充非 NaN 的行)
        df.loc[df[col].notna(), new_col] = transformed_data

        print(f"处理完毕: '{col}' -> '{new_col}' (最佳 Lambda: {best_lambda:.4f})")
        
        # 记录下来，准备删掉
        new_cols_created.append(new_col)
        cols_to_drop.append(col)

    except Exception as e:
        print(f"处理 '{col}' 时出错: {e}。该列可能不是数字，已跳过。")

# 6. 统一删除所有被成功处理过的旧列
df.drop(columns=cols_to_drop,axis=1,inplace=True)

处理完毕: 'distance_to_center' -> 'yj_distance_to_center' (最佳 Lambda: -0.9038)
处理完毕: '绿化率' -> 'yj_绿化率' (最佳 Lambda: 2.5943)
处理完毕: '燃气费' -> 'yj_燃气费' (最佳 Lambda: -0.1840)
处理完毕: '供热费' -> 'yj_供热费' (最佳 Lambda: -0.0032)
处理完毕: '客户反馈' -> 'yj_客户反馈' (最佳 Lambda: 1.1503)
处理完毕: 'Price' -> 'yj_Price' (最佳 Lambda: -0.0932)
处理完毕: '建筑面积' -> 'yj_建筑面积' (最佳 Lambda: 0.1118)
处理完毕: '套内面积' -> 'yj_套内面积' (最佳 Lambda: 0.1634)
处理完毕: '房屋总数' -> 'yj_房屋总数' (最佳 Lambda: 0.2578)
处理完毕: '楼栋总数' -> 'yj_楼栋总数' (最佳 Lambda: -0.1020)
处理完毕: '停车位' -> 'yj_停车位' (最佳 Lambda: 0.2119)
处理完毕: '停车费用' -> 'yj_停车费用' (最佳 Lambda: 0.2821)
处理完毕: '物业费' -> 'yj_物业费' (最佳 Lambda: -0.9598)
处理完毕: '容积率' -> 'yj_容积率' (最佳 Lambda: -0.6385)
处理完毕: '房龄' -> 'yj_房龄' (最佳 Lambda: -0.0386)
处理完毕: '持有年限' -> 'yj_持有年限' (最佳 Lambda: 0.1953)
处理完毕: '厨' -> 'yj_厨' (最佳 Lambda: -48.0790)
处理完毕: '室' -> 'yj_室' (最佳 Lambda: 0.6025)
处理完毕: '厅' -> 'yj_厅' (最佳 Lambda: 1.4619)
处理完毕: '户' -> 'yj_户' (最佳 Lambda: -0.6574)
处理完毕: '卫' -> 'yj_卫' (最佳 Lambda: -1.0851)
处理完毕: '梯' -> 'yj_梯' (最佳 Lambda: -1.186

In [ ]:
#-----------STAGE 6 最后清除数据------------
df.drop(['城市','区域','房屋户型','所在楼层','房屋朝向','建筑结构','梯户比例','交易时间','交易权属','上次交易','房屋用途',
        '房屋优势','核心卖点','户型介绍','周边配套','交通出行','物业类别',
        '建筑年代','绿 化 率','物 业 费','容 积 率',
        '建筑结构_comm', '供电','供水','供暖',
        '交易月份', '交易季节','yj_建筑面积'
        ],axis=1,inplace=True)

In [ ]:
#-----------STAGE 7  构建交互项 -----------
import pandas as pd
import numpy as np

# 1. 创建平方项 
df['yj_停车费用_sq'] = df['yj_停车费用'] ** 2
df['yj_户_sq'] = df['yj_户'] ** 2
df['yj_得房率_sq'] = df['yj_得房率'] ** 2

# 2. 创建交互项 
df['yj_房龄_x_yj_梯'] = df['yj_房龄'] * df['yj_梯']
df['yj_房龄_x_yj_物业费'] = df['yj_房龄'] * df['yj_物业费']
df['yj_持有年限_x_lat'] = df['yj_持有年限'] * df['lat']
df['yj_室_x_yj_总楼层_temp'] = df['yj_室'] * df['yj_总楼层_temp']
df['yj_室_x_yj_楼层位置_mapped'] = df['yj_室'] * df['yj_楼层位置_mapped']
df['yj_停车位_x_yj_楼层位置_mapped'] = df['yj_停车位'] * df['yj_楼层位置_mapped']
df['yj_卫_x_lat'] = df['yj_卫'] * df['lat']
df['yj_卫_x_yj_物业费'] = df['yj_卫'] * df['yj_物业费']
df['yj_楼栋总数_x_yj_总楼层_temp'] = df['yj_楼栋总数'] * df['yj_总楼层_temp']
df['yj_楼栋总数_x_yj_楼层位置_mapped'] = df['yj_楼栋总数'] * df['yj_楼层位置_mapped']
df['lat_x_yj_物业费'] = df['lat'] * df['yj_物业费']

print(f"--- 手动特征创建完毕。 df 维度: {df.shape} ---")
print("您现在可以运行您原来的 Cell 16 (d50b83ae) 来执行 K-Means + OLS 回归。")

--- STAGE 6 (新): 正在创建 14 个手动选择的特征... ---
--- 手动特征创建完毕。 df 维度: (137888, 132) ---
您现在可以运行您原来的 Cell 16 (d50b83ae) 来执行 K-Means + OLS 回归。


In [ ]:
#-----------STAGE 8 OLS回归+K-means超参数优化------------
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer # (用于修复 NaN 错误)
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 ---
print("--- 0. 准备数据 ---")

# 假设 'df' 是您 Cell 9 (6f0bcdf4) 运行到底的最终 DataFrame
# (它此时应该仍然包含 lon, lat)
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

print(f"    ...数据已拆分: 训练集 {train_df.shape}, 测试集 {test_df.shape}")

# --- 1. (重要) 定义逆向 Yeo-Johnson 转换 ---
# (Lambda值 -0.0932 来自您 Cell 7 (93d0a3f3) 的输出)
PRICE_LAMBDA = -0.0932

def inverse_yeojohnson(y_yj, lambda_):
    """
    Yeo-Johnson 逆向转换 (仅适用于 y_yj >= 0, lambda != 0)
    """
    # 规避 (y_yj * lambda_ + 1) 为负数
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 
    
    x = np.power(val, 1/lambda_) - 1
    return x

# --- 2. 定义评估指标 (MAE) ---
def original_price_mae_scorer(y_yj, y_pred_yj):
    # 使用 YJ 逆向转换
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    
    # 清理
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 3. 定义最终要进入模型的特征 ---
# (Cell 9 已经帮我们删除了 yj_建筑面积 和所有原始列)
TARGET = 'yj_Price' # <--- 您的目标
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' # <--- OLS 模型不直接使用它们，而是使用 cluster
]

# 获取所有剩余特征
all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns # 确保测试集也有
]

# 找出所有 float64 (需要 Impute + Scale)
numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

# 4. 定义目标 y
y_train = train_df[TARGET]

print(f"    ...准备完毕。最终 OLS 模型将使用 {len(all_model_features)} 个特征。")
print(f"    ...数值特征 (将被 Impute/Scale): {numeric_features_to_process[:3]} ...等 (共 {len(numeric_features_to_process)} 个)")

# --- Part A: 运行 K-Means 超参数搜索 (寻找 best_k) ---
print("\n--- Part A: 正在运行 K-Means 超参数搜索 (CV MAE)... ---")

def run_ols_with_k(k, train_data, model_features, numeric_features, y_target):
    """
    使用指定的k值运行K-Means，并返回OLS的6折交叉验证MAE
    """
    print(f"--- 正在测试 K = {k} ---")
    
    # 1. K-Means 聚类 (使用原始 lon/lat 拟合)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(train_data[['lon', 'lat']])
    
    # 2. 准备特征矩阵
    X_train_k = train_data[model_features].copy()
    X_train_k['cluster'] = kmeans.predict(train_data[['lon', 'lat']]).astype(str)
    
    # 3. 定义预处理器
    categorical_features_k = ['cluster'] # <--- 唯一需要 OHE 的新特征
    
    # 数值特征的子管道：捕获NaN -> 标准化
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')), # <-- 修复 NaN 错误
        ('scaler', StandardScaler())
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_k),
            ('num_pipe', numeric_transformer, numeric_features) # <-- 应用于所有 float64
        ],
        remainder='passthrough' # <-- 关键: 所有已编码的 bool/int32 特征将原样通过
    )
    
    # 4. 创建 OLS 管道
    pipeline_ols = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ])
    
    # 5. 运行交叉验证
    start_time = time.time()
    cv_scores = cross_val_score(
        pipeline_ols, X_train_k, y_target,
        cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
    )
    end_time = time.time()
    
    mean_mae = -np.mean(cv_scores)
    # (!!!) 这是您要求的实时输出 (!!!)
    print(f"K = {k}: 平均 MAE = {mean_mae:,.2f}  (耗时: {end_time - start_time:.2f} 秒)")
    
    return mean_mae

# 运行循环 (使用一个合理的步长进行搜索)
k_range = range(200,201, 10) # (K=10, 20, 30... 100)
print(f"--- (注意) 正在测试 K 范围: {list(k_range)} ---")

results = {}

for k in k_range:
    results[k] = run_ols_with_k(
        k=k,
        train_data=train_df, 
        model_features=all_model_features,
        numeric_features=numeric_features_to_process, # <-- 使用修正的列表
        y_target=y_train
    )

# --- Part B: 使用 best_k 评估最终模型 ---
print("\n--- Part B: 正在使用 Best K 评估最终 OLS 模型 ---")

# 1. 找出最佳 K 和对应的 CV MAE
best_k = min(results, key=results.get)
mae_cv = results[best_k]
print(f"    ...K-Means 优化完成。最佳 K = {best_k} (CV MAE: {mae_cv:,.2f})")

# 2. 创建最终的 K-Means
print("    ...正在构建最终管道...")
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 3. 准备最终的 X_train (带 'cluster' 特征)
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

# 4. 定义最终的 OLS 管道
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process) # <-- 使用修正的列表
    ],
    remainder='passthrough'
)
final_pipeline_ols = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', LinearRegression())
])

# 5. 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_ols.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_ols.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6. 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_ols.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_ols.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- Part C: 报告所有 OLS 关键评估指标 (包含您要的输出) ---
print("\n" + "="*50)
print("--- OLS 最终评估指标 ('price' data, MAE) ---")
print("="*50)

# (!!!) 这是您要求的表格输出 (使用 print() 来避免 'tabulate' 错误) (!!!)
print("\nK-Means 超参数搜索结果 (MAE vs. K):")
print_results = pd.DataFrame.from_dict(results, orient='index', columns=['MAE'])
print_results.index.name = 'K (聚类数)'
print(print_results)
# (!!!) 输出结束 (!!!)

print("\n最终模型性能:")
print(f"最佳 K-Means 聚类数: {best_k}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

--- 0. 准备数据 ---
    ...数据已拆分: 训练集 (103871, 132), 测试集 (34017, 132)
    ...准备完毕。最终 OLS 模型将使用 127 个特征。
    ...数值特征 (将被 Impute/Scale): ['核心卖点_长度', 'yj_distance_to_center', 'yj_绿化率'] ...等 (共 39 个)

--- Part A: 正在运行 K-Means 超参数搜索 (CV MAE)... ---
--- (注意) 正在测试 K 范围: [200] ---
--- 正在测试 K = 200 ---
K = 200: 平均 MAE = 441,701.39  (耗时: 13.44 秒)

--- Part B: 正在使用 Best K 评估最终 OLS 模型 ---
    ...K-Means 优化完成。最佳 K = 200 (CV MAE: 441,701.39)
    ...正在构建最终管道...
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- OLS 最终评估指标 ('price' data, MAE) ---

K-Means 超参数搜索结果 (MAE vs. K):
                   MAE
K (聚类数)               
200      441701.390945

最终模型性能:
最佳 K-Means 聚类数: 200
In-sample MAE (训练集):      439,853.56
Out-of-sample MAE (20%验证集): 440,215.43
Cross-validation MAE (6-fold): 441,701.39


In [ ]:
# --- 生成测试集预测 ---
print("\n--- Part D: 正在生成测试集预测... ---")

# 准备 X_test
X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)

# (final_pipeline_ols 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_ols.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
# 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_price_ols.csv', index=False)
print("已生成对测试集的预测文件 'prediction_price_ols.csv'")


--- Part D: 正在生成测试集预测... ---
已生成对测试集的预测文件 'prediction_price_ols.csv'


In [ ]:
#-----------STAGE 9 Lasso回归+K-means超参数优化------------
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LassoCV # <--- (新) 导入 LassoCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer 
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 (与 OLS 单元格相同) ---
print("--- 0. 准备数据 (用于 Lasso)... ---")
# (我们使用的是 Cell 36 运行后内存中的 'df')
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 1. 定义逆向转换和评估器 (与 OLS 单元格相同) ---
PRICE_LAMBDA = -0.0932

def inverse_yeojohnson(y_yj, lambda_):
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 # 规避负数
    x = np.power(val, 1/lambda_) - 1
    return x

def original_price_mae_scorer(y_yj, y_pred_yj):
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 2. 定义特征 (与 OLS 单元格相同) ---
TARGET = 'yj_Price'
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' 
]

all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns
]

numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

y_train = train_df[TARGET]

print(f"    ...准备完毕。Lasso 模型将使用 {len(all_model_features)} 个特征。")

# --- 3. K-Means 特征工程 (使用 OLS 找到的最佳 K=200) ---
print(f"--- 3. 正在应用 K-Means (K=200)... ---")
best_k = 200 # <-- 来自 OLS 单元格 的结果
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 准备 X_train 和 X_test
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)

# --- 4. 定义预处理器 (与 OLS 单元格相同) ---
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process)
    ],
    remainder='passthrough'
)

# --- 5. (新) 定义 LassoCV 回归管道 ---
print("--- 5. 正在构建 LassoCV 管道... ---")

# LassoCV 会自动使用 6 折交叉验证 (cv=kf_6) 寻找最佳的 alpha (正则化强度)
lasso_model = LassoCV(cv=kf_6, n_jobs=-1, random_state=42)

# 组装最终管道
final_pipeline_lasso = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', lasso_model) # <-- (新) 将回归器替换为 LassoCV
])

# --- 6. (新) 评估 Lasso 模型 ---

# 6.1 计算 Cross-validation MAE
print("    ...正在计算 Cross-validation MAE (6-fold)...")
start_time_cv = time.time()
cv_scores = cross_val_score(
    final_pipeline_lasso, X_train_final, y_train,
    cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
)
mae_cv = -np.mean(cv_scores)
print(f"    (CV 耗时: {time.time() - start_time_cv:.2f} 秒)")

# 6.2 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_lasso.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_lasso.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6.3 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_lasso.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_lasso.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- 7. (新) 报告 Lasso 评估指标 ---
print("\n" + "="*50)
print("--- LassoCV 最终评估指标 ('price' data, MAE) ---")
print("="*50)
print(f"最佳 K-Means 聚类数: {best_k}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

# --- 8. (新) 生成 Lasso 预测文件 ---
print("\n--- Part D: 正在生成 Lasso 测试集预测... ---")

# (final_pipeline_lasso 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_lasso.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_price_lasso.csv', index=False) # <-- (新) 命名
print("已生成对测试集的预测文件 'prediction_price_lasso.csv'")

--- 0. 准备数据 (用于 Lasso)... ---
    ...准备完毕。Lasso 模型将使用 127 个特征。
--- 3. 正在应用 K-Means (K=200)... ---
--- 5. 正在构建 LassoCV 管道... ---
    ...正在计算 Cross-validation MAE (6-fold)...
    (CV 耗时: 78.97 秒)
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- LassoCV 最终评估指标 ('price' data, MAE) ---
最佳 K-Means 聚类数: 200
In-sample MAE (训练集):      486,192.60
Out-of-sample MAE (20%验证集): 516,860.98
Cross-validation MAE (6-fold): 504,548.65

--- Part D: 正在生成 Lasso 测试集预测... ---
已生成对测试集的预测文件 'prediction_price_lasso.csv'


In [ ]:
#-----------STAGE 10 Ridge回归+K-means超参数优化------------
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import RidgeCV # <--- (新) 导入 RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.impute import SimpleImputer 
import time
import warnings

warnings.filterwarnings('ignore')

# --- 0. 准备数据 (与 OLS/Lasso 单元格相同) ---
print("--- 0. 准备数据 (用于 Ridge)... ---")
# (我们使用的是 Cell 36 运行后内存中的 'df')
train_df = df[df['train'] == 1].copy()
test_df = df[df['train'] == 0].copy()

# --- 1. 定义逆向转换和评估器 (与 OLS/Lasso 单元格相同) ---
PRICE_LAMBDA = -0.0932

def inverse_yeojohnson(y_yj, lambda_):
    val = y_yj * lambda_ + 1
    val[val < 0] = 0.000001 # 规避负数
    x = np.power(val, 1/lambda_) - 1
    return x

def original_price_mae_scorer(y_yj, y_pred_yj):
    y_orig = inverse_yeojohnson(y_yj, PRICE_LAMBDA)
    y_pred_orig = inverse_yeojohnson(y_pred_yj, PRICE_LAMBDA)
    y_pred_orig = np.nan_to_num(y_pred_orig, nan=0.0, posinf=np.finfo(np.float64).max)
    y_pred_orig = np.clip(y_pred_orig, 0, None)
    return mean_absolute_error(y_orig, y_pred_orig)

custom_mae_scorer = make_scorer(original_price_mae_scorer, greater_is_better=False)
kf_6 = KFold(n_splits=6, shuffle=True, random_state=42)

# --- 2. 定义特征 (与 OLS/Lasso 单元格相同) ---
TARGET = 'yj_Price'
COLS_TO_DROP_FINAL = [
    'ID', 'train', TARGET,
    'lon', 'lat' 
]

all_model_features = [
    col for col in train_df.columns 
    if col not in COLS_TO_DROP_FINAL 
    and col in test_df.columns
]

numeric_features_to_process = [
    col for col in all_model_features 
    if train_df[col].dtype == 'float64'
]

y_train = train_df[TARGET]

print(f"    ...准备完毕。Ridge 模型将使用 {len(all_model_features)} 个特征。")

# --- 3. K-Means 特征工程 (使用 OLS 找到的最佳 K=200) ---
print(f"--- 3. 正在应用 K-Means (K=200)... ---")
best_k = 200 # <-- 来自 OLS 单元格 的结果
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(train_df[['lon', 'lat']])

# 准备 X_train 和 X_test
X_train_final = train_df[all_model_features].copy()
X_train_final['cluster'] = kmeans_final.predict(train_df[['lon', 'lat']]).astype(str)

X_test_final = test_df[all_model_features].copy()
X_test_final['cluster'] = kmeans_final.predict(test_df[['lon', 'lat']]).astype(str)

# --- 4. 定义预处理器 (与 OLS/Lasso 单元格相同) ---
categorical_features_final = ['cluster']

final_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

final_preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_k', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features_final),
        ('num_pipe', final_numeric_transformer, numeric_features_to_process)
    ],
    remainder='passthrough'
)

# --- 5. (新) 定义 RidgeCV 回归管道 ---
print("--- 5. 正在构建 RidgeCV 管道... ---")

# (新) RidgeCV 通常需要我们提供一个 alpha 列表进行测试
# 这是一个对数间隔的列表 (例如: 0.1, 1.0, 10.0, 100.0)
alphas_to_test = np.logspace(-2, 4, 10) 
print(f"    ...将测试 {len(alphas_to_test)} 个 Alpha 值 (从 {alphas_to_test[0]} 到 {alphas_to_test[-1]})")

# RidgeCV 会自动使用 6 折交叉验证 (cv=kf_6) 从列表中寻找最佳 alpha
ridge_model = RidgeCV(alphas=alphas_to_test, cv=kf_6)

# 组装最终管道
final_pipeline_ridge = Pipeline(steps=[
    ('preprocessor', final_preprocessor),
    ('regressor', ridge_model) # <-- (新) 将回归器替换为 RidgeCV
])

# --- 6. (新) 评估 Ridge 模型 ---

# 6.1 计算 Cross-validation MAE
# (注意: 这里我们再次对 *整个* 管道进行 CV，以获得最终的性能分数)
print("    ...正在计算 Cross-validation MAE (6-fold)...")
start_time_cv = time.time()
cv_scores = cross_val_score(
    final_pipeline_ridge, X_train_final, y_train,
    cv=kf_6, scoring=custom_mae_scorer, n_jobs=-1
)
mae_cv = -np.mean(cv_scores)
print(f"    (CV 耗时: {time.time() - start_time_cv:.2f} 秒)")

# 6.2 计算 Out-of-sample (local validation) MAE
print("    ...正在计算 Out-of-Sample (本地验证) MAE...")
X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    X_train_final, y_train, test_size=0.2, random_state=42
)
final_pipeline_ridge.fit(X_train_local, y_train_local)
y_pred_val_log = final_pipeline_ridge.predict(X_val_local)
mae_out_of_sample = original_price_mae_scorer(y_val_local, y_pred_val_log)

# 6.3 计算 In-sample MAE
print("    ...正在计算 In-Sample MAE...")
final_pipeline_ridge.fit(X_train_final, y_train)
y_pred_train_log = final_pipeline_ridge.predict(X_train_final)
mae_in_sample = original_price_mae_scorer(y_train, y_pred_train_log)

# --- 7. (新) 报告 Ridge 评估指标 ---
print("\n" + "="*50)
print("--- RidgeCV 最终评估指标 ('price' data, MAE) ---")
print("="*50)
print(f"最佳 K-Means 聚类数: {best_k}")
# (新) 打印 Ridge 找到的最佳 alpha
best_alpha_ridge = final_pipeline_ridge.named_steps['regressor'].alpha_
print(f"Ridge 找到的最佳 Alpha: {best_alpha_ridge}")
print(f"In-sample MAE (训练集):      {mae_in_sample:,.2f}")
print(f"Out-of-sample MAE (20%验证集): {mae_out_of_sample:,.2f}")
print(f"Cross-validation MAE (6-fold): {mae_cv:,.2f}")
print("="*50)

# --- 8. (新) 生成 Ridge 预测文件 ---
print("\n--- Part D: 正在生成 Ridge 测试集预测... ---")

# (final_pipeline_ridge 已经用 100% 训练数据拟合过了)
test_pred_log = final_pipeline_ridge.predict(X_test_final)
test_pred_price = inverse_yeojohnson(test_pred_log, PRICE_LAMBDA)
 
# 创建提交文件
submission = pd.DataFrame({'ID': test_df['ID'], 'Price': test_pred_price})
submission.to_csv('prediction_price_ridge.csv', index=False) # <-- (新) 命名
print("已生成对测试集的预测文件 'prediction_price_ridge.csv'")

--- 0. 准备数据 (用于 Ridge)... ---
    ...准备完毕。Ridge 模型将使用 127 个特征。
--- 3. 正在应用 K-Means (K=200)... ---
--- 5. 正在构建 RidgeCV 管道... ---
    ...将测试 10 个 Alpha 值 (从 0.01 到 10000.0)
    ...正在计算 Cross-validation MAE (6-fold)...
    (CV 耗时: 89.72 秒)
    ...正在计算 Out-of-Sample (本地验证) MAE...
    ...正在计算 In-Sample MAE...

--- RidgeCV 最终评估指标 ('price' data, MAE) ---
最佳 K-Means 聚类数: 200
Ridge 找到的最佳 Alpha: 0.046415888336127774
In-sample MAE (训练集):      439,900.89
Out-of-sample MAE (20%验证集): 440,282.84
Cross-validation MAE (6-fold): 441,745.92

--- Part D: 正在生成 Ridge 测试集预测... ---
已生成对测试集的预测文件 'prediction_price_ridge.csv'
